In [1]:
# ============================================
# MODULE 3: NEWS CATEGORY DETECTION
# ============================================

import pandas as pd
import re
import string
import nltk
import pickle
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Download stopwords if not already downloaded
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('stopwords')
    nltk.download('punkt')

stop_words = set(stopwords.words("english"))

In [ ]:
# ============================================
# STEP 1: LOAD AND COMBINE ALL DATASETS
# ============================================

print("="*60)
print("LOADING AND COMBINING DATASETS")
print("="*60)

datasets = []

# 1. Load fake.csv (Fake News)
try:
    fake = pd.read_csv("../datasets/fake.csv")
    # For category detection, we need to assign categories
    # You can assign based on subject or create a general category
    if 'subject' in fake.columns:
        # Map subjects to categories
        subject_map = {
            'politics': 'Politics',
            'worldnews': 'World',
            'left-news': 'Politics',
            'News': 'General'
        }
        fake['category'] = fake['subject'].map(lambda x: subject_map.get(x, 'General'))
    else:
        fake['category'] = 'General'
    
    fake['source'] = 'fake'
    datasets.append(fake)
    print(f"✓ Loaded fake.csv: {fake.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load fake.csv: {e}")

# 2. Load True.csv (True News)
try:
    true = pd.read_csv("../datasets/True.csv")
    if 'subject' in true.columns:
        subject_map = {
            'politicsNews': 'Politics',
            'worldnews': 'World',
        }
        true['category'] = true['subject'].map(lambda x: subject_map.get(x, 'General'))
    else:
        true['category'] = 'General'
    
    true['source'] = 'true'
    datasets.append(true)
    print(f"✓ Loaded True.csv: {true.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load True.csv: {e}")

# 3. Load AG_Train.csv (AG News Training)
try:
    ag_train = pd.read_csv("../datasets/AG_Train.csv")
    print(f"AG_Train columns: {ag_train.columns.tolist()}")
    
    # Get category from Class Index
    if 'Class Index' in ag_train.columns:
        category_map = {
            1: "World",
            2: "Sports",
            3: "Business",
            4: "Sci/Tech"
        }
        ag_train['category'] = ag_train['Class Index'].map(category_map)
    elif 'class' in ag_train.columns:
        category_map = {
            1: "World",
            2: "Sports", 
            3: "Business",
            4: "Sci/Tech"
        }
        ag_train['category'] = ag_train['class'].map(category_map)
    elif 'label' in ag_train.columns:
        category_map = {
            1: "World",
            2: "Sports",
            3: "Business",
            4: "Sci/Tech"
        }
        ag_train['category'] = ag_train['label'].map(category_map)
    else:
        first_col = ag_train.columns[0]
        category_map = {
            1: "World",
            2: "Sports",
            3: "Business",
            4: "Sci/Tech"
        }
        ag_train['category'] = ag_train[first_col].map(category_map)
    
    # Combine Title and Description
    if 'Title' in ag_train.columns and 'Description' in ag_train.columns:
        ag_train['text'] = ag_train['Title'].fillna('') + ' ' + ag_train['Description'].fillna('')
    elif 'title' in ag_train.columns and 'description' in ag_train.columns:
        ag_train['text'] = ag_train['title'].fillna('') + ' ' + ag_train['description'].fillna('')
    elif len(ag_train.columns) >= 3:
        col_names = ag_train.columns.tolist()
        ag_train['text'] = ag_train[col_names[1]].fillna('') + ' ' + ag_train[col_names[2]].fillna('')
    else:
        text_col = ag_train.columns[1] if len(ag_train.columns) > 1 else ag_train.columns[0]
        ag_train['text'] = ag_train[text_col]
    
    ag_train['source'] = 'AG_Train'
    datasets.append(ag_train)
    print(f"✓ Loaded AG_Train.csv: {ag_train.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load AG_Train.csv: {e}")

# 4. Load AG_Test.csv (AG News Test)
try:
    ag_test = pd.read_csv("../datasets/AG_Test.csv")
    print(f"AG_Test columns: {ag_test.columns.tolist()}")
    
    if 'Class Index' in ag_test.columns:
        category_map = {
            1: "World",
            2: "Sports",
            3: "Business",
            4: "Sci/Tech"
        }
        ag_test['category'] = ag_test['Class Index'].map(category_map)
    elif 'class' in ag_test.columns:
        category_map = {
            1: "World",
            2: "Sports",
            3: "Business",
            4: "Sci/Tech"
        }
        ag_test['category'] = ag_test['class'].map(category_map)
    elif 'label' in ag_test.columns:
        category_map = {
            1: "World",
            2: "Sports",
            3: "Business",
            4: "Sci/Tech"
        }
        ag_test['category'] = ag_test['label'].map(category_map)
    else:
        first_col = ag_test.columns[0]
        category_map = {
            1: "World",
            2: "Sports",
            3: "Business",
            4: "Sci/Tech"
        }
        ag_test['category'] = ag_test[first_col].map(category_map)
    
    if 'Title' in ag_test.columns and 'Description' in ag_test.columns:
        ag_test['text'] = ag_test['Title'].fillna('') + ' ' + ag_test['Description'].fillna('')
    elif 'title' in ag_test.columns and 'description' in ag_test.columns:
        ag_test['text'] = ag_test['title'].fillna('') + ' ' + ag_test['description'].fillna('')
    elif len(ag_test.columns) >= 3:
        col_names = ag_test.columns.tolist()
        ag_test['text'] = ag_test[col_names[1]].fillna('') + ' ' + ag_test[col_names[2]].fillna('')
    else:
        text_col = ag_test.columns[1] if len(ag_test.columns) > 1 else ag_test.columns[0]
        ag_test['text'] = ag_test[text_col]
    
    ag_test['source'] = 'AG_Test'
    datasets.append(ag_test)
    print(f"✓ Loaded AG_Test.csv: {ag_test.shape[0]:,} rows")
except Exception as e:
    print(f"✗ Could not load AG_Test.csv: {e}")

# Combine all datasets
df = pd.concat(datasets, ignore_index=True)

print("-"*60)
print(f"Total combined rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")

# Check category distribution
print("\nCategory Distribution:")
print(df['category'].value_counts())

LOADING AND COMBINING DATASETS
✓ Loaded fake.csv: 23,481 rows
✓ Loaded True.csv: 21,417 rows
AG_Train columns: ['Class Index', 'Title', 'Description']
✓ Loaded AG_Train.csv: 120,000 rows
AG_Test columns: ['Class Index', 'Title', 'Description']
✓ Loaded AG_Test.csv: 7,600 rows
------------------------------------------------------------
Total combined rows: 172,498
Columns: ['title', 'text', 'subject', 'date', 'category', 'source', 'Class Index', 'Title', 'Description']

Category Distribution:
category
World       42045
Business    31900
Sci/Tech    31900
Sports      31900
Politics    22572
General     12181
Name: count, dtype: int64


In [3]:
# ============================================
# STEP 2: TEXT CLEANING
# ============================================

def clean_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return ""
    
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

In [4]:
# ============================================
# STEP 3: CLEAN THE TEXT DATA
# ============================================

print("\n" + "="*60)
print("CLEANING TEXT DATA")
print("="*60)

# Check if 'text' column exists, if not create it
if 'text' not in df.columns:
    if 'title' in df.columns and 'description' in df.columns:
        df['text'] = df['title'].fillna('') + ' ' + df['description'].fillna('')
    elif 'title' in df.columns:
        df['text'] = df['title'].fillna('')
    else:
        # Use first available text column
        text_col = df.columns[0]
        df['text'] = df[text_col]

df["clean_text"] = df["text"].apply(clean_text)

# Remove empty text after cleaning
before = len(df)
df = df[df['clean_text'].str.len() > 0]
after = len(df)
print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning: {after:,}")
print(f"Rows removed: {before - after:,}")

# Remove duplicates
before = len(df)
df = df.drop_duplicates(subset=['clean_text'])
after = len(df)
print(f"\nRows before removing duplicates: {before:,}")
print(f"Rows after removing duplicates: {after:,}")
print(f"Duplicates removed: {before - after:,}")


CLEANING TEXT DATA
Rows before cleaning: 172,498
Rows after cleaning: 171,782
Rows removed: 716

Rows before removing duplicates: 171,782
Rows after removing duplicates: 165,835
Duplicates removed: 5,947


In [5]:
# ============================================
# STEP 4: TRAIN-TEST SPLIT
# ============================================

print("\n" + "="*60)
print("TRAIN-TEST SPLIT")
print("="*60)

X = df["clean_text"]
y = df["category"]

# Encode categories to numbers
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"Training set size: {len(X_train):,}")
print(f"Testing set size: {len(X_test):,}")
print(f"Train/Test split: {len(X_train)/len(X)*100:.1f}% / {len(X_test)/len(X)*100:.1f}%")

print("\nTraining set category distribution:")
train_categories = pd.Series(le.inverse_transform(y_train)).value_counts()
for cat, count in train_categories.items():
    print(f"  {cat}: {count:,} ({count/len(y_train)*100:.2f}%)")

print("\nTesting set category distribution:")
test_categories = pd.Series(le.inverse_transform(y_test)).value_counts()
for cat, count in test_categories.items():
    print(f"  {cat}: {count:,} ({count/len(y_test)*100:.2f}%)")


TRAIN-TEST SPLIT
Training set size: 132,668
Testing set size: 33,167
Train/Test split: 80.0% / 20.0%

Training set category distribution:
  World: 33,443 (25.21%)
  Sports: 25,470 (19.20%)
  Business: 25,451 (19.18%)
  Sci/Tech: 25,419 (19.16%)
  Politics: 14,607 (11.01%)
  General: 8,278 (6.24%)

Testing set category distribution:
  World: 8,361 (25.21%)
  Sports: 6,368 (19.20%)
  Business: 6,363 (19.18%)
  Sci/Tech: 6,355 (19.16%)
  Politics: 3,651 (11.01%)
  General: 2,069 (6.24%)


In [6]:
# ============================================
# STEP 5: TF-IDF VECTORIZATION
# ============================================

print("\n" + "="*60)
print("TF-IDF VECTORIZATION")
print("="*60)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.9
)

print("Fitting vectorizer on training data...")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Training data shape: {X_train_tfidf.shape}")
print(f"Testing data shape: {X_test_tfidf.shape}")
print(f"Number of features: {len(vectorizer.get_feature_names_out()):,}")


TF-IDF VECTORIZATION
Fitting vectorizer on training data...
Training data shape: (132668, 5000)
Testing data shape: (33167, 5000)
Number of features: 5,000


In [7]:
# ============================================
# STEP 6: TRAIN MODELS
# ============================================

print("\n" + "="*60)
print("TRAINING MODELS")
print("="*60)

models = {
    'Naive Bayes': MultinomialNB(),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=25,
        n_jobs=-1,
        random_state=42
    )
}

best_accuracy = 0
best_model = None
best_model_name = ""
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    try:
        model.fit(X_train_tfidf, y_train)
        predictions = model.predict(X_test_tfidf)
        accuracy = accuracy_score(y_test, predictions)
        results[name] = accuracy
        print(f"  ✓ {name} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
        
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model = model
            best_model_name = name
    except Exception as e:
        print(f"  ✗ {name} failed: {e}")

print("\n" + "-"*60)
print("MODEL COMPARISON RESULTS:")
print("-"*60)
sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
for rank, (name, acc) in enumerate(sorted_results, 1):
    print(f"  #{rank}. {name}: {acc:.4f} ({acc*100:.2f}%)")

print(f"\n✓ Best Model: {best_model_name} with {best_accuracy*100:.2f}% accuracy")
model = best_model


TRAINING MODELS

Training Naive Bayes...
  ✓ Naive Bayes Accuracy: 0.8613 (86.13%)

Training Random Forest...
  ✓ Random Forest Accuracy: 0.8114 (81.14%)

------------------------------------------------------------
MODEL COMPARISON RESULTS:
------------------------------------------------------------
  #1. Naive Bayes: 0.8613 (86.13%)
  #2. Random Forest: 0.8114 (81.14%)

✓ Best Model: Naive Bayes with 86.13% accuracy


In [8]:
# ============================================
# STEP 7: EVALUATE BEST MODEL
# ============================================

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

predictions = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Classification Report with category names
category_names = le.classes_
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=category_names))


MODEL EVALUATION
Accuracy: 0.8613 (86.13%)

Classification Report:
              precision    recall  f1-score   support

    Business       0.86      0.83      0.84      6363
     General       0.71      0.89      0.79      2069
    Politics       0.84      0.73      0.78      3651
    Sci/Tech       0.85      0.83      0.84      6355
      Sports       0.93      0.96      0.94      6368
       World       0.88      0.89      0.88      8361

    accuracy                           0.86     33167
   macro avg       0.84      0.85      0.85     33167
weighted avg       0.86      0.86      0.86     33167



In [9]:
# ============================================
# STEP 8: PREDICTION FUNCTION
# ============================================

def predict_category(news_text):
    """
    Predict the category of a news article
    
    Parameters:
    -----------
    news_text : str
        Text of the news article
        
    Returns:
    --------
    dict : Prediction result with category and confidence
    """
    if isinstance(news_text, str):
        news_text = [news_text]
    
    # Clean the text
    cleaned_text = [clean_text(text) for text in news_text]
    
    # Vectorize
    vectors = vectorizer.transform(cleaned_text)
    
    # Predict
    predictions = model.predict(vectors)
    probabilities = model.predict_proba(vectors)
    
    results = []
    for i, (pred, probs) in enumerate(zip(predictions, probabilities)):
        category = le.inverse_transform([pred])[0]
        confidence = max(probs)
        
        # Get all categories with probabilities
        all_categories = {cat: prob for cat, prob in zip(le.classes_, probs)}
        sorted_categories = sorted(all_categories.items(), key=lambda x: x[1], reverse=True)
        
        result = {
            'text': news_text[i] if isinstance(news_text, list) else news_text,
            'category': category,
            'confidence': confidence,
            'all_categories': sorted_categories
        }
        results.append(result)
    
    return results[0] if len(results) == 1 else results

# Test the prediction function
print("\n" + "="*60)
print("TESTING PREDICTION")
print("="*60)

test_articles = [
    "Pakistan won the cricket world cup final",
    "Oil prices increased due to rising global demand",
    "Apple launched a new AI powered iPhone",
    "World leaders gathered for climate summit"
]

for article in test_articles:
    result = predict_category(article)
    print(f"\nArticle: {article}")
    print(f"Predicted Category: {result['category']}")
    print(f"Confidence: {result['confidence']:.2%}")
    print("Top Categories:")
    for cat, prob in result['all_categories'][:3]:
        print(f"  {cat}: {prob:.2%}")
    print("-"*40)


TESTING PREDICTION

Article: Pakistan won the cricket world cup final
Predicted Category: Sports
Confidence: 99.21%
Top Categories:
  Sports: 99.21%
  World: 0.75%
  Sci/Tech: 0.02%
----------------------------------------

Article: Oil prices increased due to rising global demand
Predicted Category: Business
Confidence: 98.08%
Top Categories:
  Business: 98.08%
  World: 1.30%
  Sci/Tech: 0.53%
----------------------------------------

Article: Apple launched a new AI powered iPhone
Predicted Category: Sci/Tech
Confidence: 91.90%
Top Categories:
  Sci/Tech: 91.90%
  Business: 5.34%
  World: 0.92%
----------------------------------------

Article: World leaders gathered for climate summit
Predicted Category: World
Confidence: 69.61%
Top Categories:
  World: 69.61%
  Politics: 15.54%
  Sci/Tech: 6.44%
----------------------------------------


In [10]:
# ============================================
# STEP 9: SAVE MODEL AND VECTORIZER
# ============================================

print("\n" + "="*60)
print("SAVING MODEL")
print("="*60)

# Save model
with open("category_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("✓ Model saved to: category_model.pkl")

# Save vectorizer
with open("category_tfidf.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
print("✓ Vectorizer saved to: category_tfidf.pkl")

# Save label encoder
with open("category_label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
print("✓ Label encoder saved to: category_label_encoder.pkl")

print("\n" + "="*60)
print("MODULE 3 COMPLETE!")
print("="*60)


SAVING MODEL
✓ Model saved to: category_model.pkl
✓ Vectorizer saved to: category_tfidf.pkl
✓ Label encoder saved to: category_label_encoder.pkl

MODULE 3 COMPLETE!
